# Bài 6
Đây là notebook chứa mã nguồn đầy đủ của bài 6.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def _entropy_weights(matrix):
    m, n = matrix.shape
    col_sums = matrix.sum(axis=0) + 1e-9
    P = matrix / col_sums
    P = np.clip(P, 1e-9, 1)
    E = -1 / np.log(m) * np.sum(P * np.log(P), axis=0)
    d = 1 - E
    return d / (d.sum() + 1e-9)

In [ ]:
def solve_bai06(data_dir=None, w_manual=None, weight_mode=0):
    data = get_data(data_dir)
    regions  = data.regions_names_vi.tolist()
    
    # Extract features matching the Bai 6 requirements
    # PDF: GRDP/người, FDI, Digital Index, AI Readiness, LĐ ĐT, R&D/GRDP, Internet, Gini
    # Let's use the provided DataLoader X_regions.
    # In data_loader.py: X_regions = [GRDP_capita, Digital, AI, Labor, R&D, Gini]
    # We will use exactly these 6.
    X = data.X_regions[:, [0, 2, 3, 4, 5, 7]].astype(float)
    
    # is_benefit
    is_benefit = [True, True, True, True, True, False]
    
    if weight_mode < 0.5:
        w = _entropy_weights(X)
    else:
        w = np.array(w_manual if w_manual else [0.20, 0.20, 0.20, 0.15, 0.15, 0.10], float)

    # Vector normalization
    norm = np.sqrt((X**2).sum(axis=0))
    R = X / (norm + 1e-9)

    # Weighted normalized matrix
    V = R * w

    # Ideal and anti-ideal
    ideal      = np.zeros(6)
    anti_ideal = np.zeros(6)
    for j in range(6):
        if is_benefit[j]:
            ideal[j]      = V[:, j].max()
            anti_ideal[j] = V[:, j].min()
        else:
            ideal[j]      = V[:, j].min()
            anti_ideal[j] = V[:, j].max()

    # Distances
    D_plus  = np.sqrt(((V - ideal)**2).sum(axis=1))
    D_minus = np.sqrt(((V - anti_ideal)**2).sum(axis=1))

    # Closeness coefficient
    C = D_minus / (D_plus + D_minus + 1e-9)
    ranking = np.argsort(-C)

    result = []
    for rank, i in enumerate(ranking):
        result.append({
            'region_name_vi': regions[i],
            'TOPSIS':         round(float(C[i]), 4),
            'D_plus':         round(float(D_plus[i]), 4),
            'D_minus':        round(float(D_minus[i]), 4),
            'rank':           rank + 1,
        })

    # Sensitivity AI (index 2 is AI Readiness)
    sensitivity = {}
    for w_ai_val in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
        w_temp = w.copy()
        w_temp[2] = w_ai_val
        w_temp = w_temp / w_temp.sum()
        V_temp = R * w_temp
        
        ideal_t = np.zeros(6)
        anti_ideal_t = np.zeros(6)
        for j in range(6):
            if is_benefit[j]:
                ideal_t[j] = V_temp[:, j].max()
                anti_ideal_t[j] = V_temp[:, j].min()
            else:
                ideal_t[j] = V_temp[:, j].min()
                anti_ideal_t[j] = V_temp[:, j].max()
                
        D_plus_t = np.sqrt(((V_temp - ideal_t)**2).sum(axis=1))
        D_minus_t = np.sqrt(((V_temp - anti_ideal_t)**2).sum(axis=1))
        C_t = D_minus_t / (D_plus_t + D_minus_t + 1e-9)
        sensitivity[str(round(w_ai_val, 2))] = {regions[i]: round(float(C_t[i]), 4) for i in range(len(regions))}

    # Simple Weighted Sum (AHP-like)
    score_ahp = (R * w).sum(axis=1)
    rank_ahp_order = np.argsort(-score_ahp)
    ahp_ranking_positions = [0]*len(regions)
    for rank_idx, i in enumerate(rank_ahp_order):
        ahp_ranking_positions[i] = rank_idx + 1
        
    ranks_comparison = []
    topsis_ranking_positions = [0]*len(regions)
    for rank_idx, i in enumerate(ranking):
        topsis_ranking_positions[i] = rank_idx + 1
        
    for i in range(len(regions)):
        ranks_comparison.append({
            'Vùng': regions[i],
            'Hạng TOPSIS': topsis_ranking_positions[i],
            'Hạng AHP đơn giản': ahp_ranking_positions[i],
            'Chênh lệch': abs(topsis_ranking_positions[i] - ahp_ranking_positions[i])
        })

    return {
        'ranking':    result,
        'weights':    w.tolist(),
        'ideal':      ideal.tolist(),
        'anti_ideal': anti_ideal.tolist(),
        'closeness':  {regions[i]: round(float(C[i]), 4) for i in range(len(regions))},
        'sensitivity': sensitivity,
        'ranks_comparison': ranks_comparison,
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai06()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())